# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR² dataset of protein data from human extracellular vesicles using the `mlcroissant` library.

### Dataset Source
The dataset metadata and structure are described via a [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) URL.

We will:
- Load dataset metadata and examine the available record sets and fields (referenced by their `@id`s)
- Extract records from a key record set
- Perform basic processing and exploratory data analysis (EDA)
- Visualize selected aspects of the data

_Note: All entities (record sets, fields, etc.) are referenced by their `@id` fields per best practice._

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

In [ ]:
# List all record sets by their @id, and explore their fields/columns

record_set_ids = []
print('Available record sets:')
for rs in metadata.record_sets:
    print(f"- Record Set Name: {rs.name}, @id: {rs.id}")
    record_set_ids.append(rs.id)
    # Print out the fields/columns @id
    print("  Fields/Columns (@id):")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id})")
    print()
# We'll pick the main protein recordset for demonstration (first one in list)
main_record_set_id = record_set_ids[0] if record_set_ids else None
print(f"Chosen main record set: {main_record_set_id}")

## 3. Data Extraction
Load records from the main record set as a DataFrame, referencing all entities by their `@id`s.

In [ ]:
# Load all record sets into dataframes
dataframes = {}

for record_set_id in record_set_ids:
    # Get records using the @id
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

if main_record_set_id and main_record_set_id in dataframes:
    print(f"Columns in main record set ({main_record_set_id}):\n{dataframes[main_record_set_id].columns.tolist()}\n")
    display(dataframes[main_record_set_id].head())
else:
    print('No data found in record sets.')

## 4. Exploratory Data Analysis (EDA)

We'll perform basic filtering, normalization and grouping on numeric fields.

Please update the variables below to match the appropriate field `@id`s shown above. Here, an example field like `cr:peptide_count` is used as a numeric field, and `cr:modification` as a grouping field. Adjust based on the specific field list output above.

In [ ]:
# Select numeric and group fields using their @id
# Example: numeric_field = 'cr:peptide_count', group_field = 'cr:modification'
main_df = dataframes.get(main_record_set_id, pd.DataFrame())

# User should check available columns printed above and adjust these as needed.
# For demo, we auto-select a numeric and a group field:

numeric_field = None
group_field = None
for col in main_df.columns:
    if numeric_field is None and main_df[col].dtype in ['float64','int64','int32','float32']:
        numeric_field = col
    # Pick a non-numeric field for grouping
    if group_field is None and main_df[col].dtype == 'object':
        group_field = col

print(f"Numeric field for EDA: {numeric_field}\nGrouping field: {group_field}\n")

# Remove missing/NaN values for the selected numeric field
filtered_df = main_df.copy()
if numeric_field is not None:
    filtered_df = filtered_df[filtered_df[numeric_field].notnull()]
    threshold = filtered_df[numeric_field].quantile(0.5)  # median as a demo threshold
    filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (median): [{len(filtered_df)} records]")
    display(filtered_df[[numeric_field]].head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"First few normalized records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping (if group_field is present)
    if group_field and group_field in filtered_df.columns:
        grouped_df = (
            filtered_df.groupby(group_field)[numeric_field]
            .mean()
            .reset_index()
            .sort_values(numeric_field, ascending=False)
        )
        print(f"Average {numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print("Could not determine a numeric field for EDA. Please check available columns!")

## 5. Visualization
Visualize distribution of the selected numeric field, and its grouping if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and not filtered_df.empty:
    plt.figure(figsize=(8,5))
    sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
    plt.title(f"{numeric_field} Distribution (filtered)")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if group_field and group_field in filtered_df.columns:
        # Show top 10 groups by mean value (if enough unique groups)
        grouped = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
        plt.figure(figsize=(10,5))
        sns.barplot(x=grouped.index, y=grouped.values)
        plt.title(f"Top {group_field} categories by mean {numeric_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we've:
- Loaded metadata and explored key record sets and their column/field `@id`s
- Loaded main protein data as a DataFrame and performed basic exploratory analysis
- Filtered and normalized a numeric field, demonstrating practical data manipulation referencing Croissant schema entities
- Visualized distribution of numeric data and group means

Further work can include deeper exploration of protein abundance patterns and additional visualizations, such as heatmaps or sample comparisons.

Remember to always reference dataset structure using record set, field, and column `@id`s for reproducibility and schema alignment.